# Tutorial 1: Quickstart Scientific Taste

Estimated time: 20-30 minutes

## Prerequisites
No optional dependencies required.

## Learning aims
- Primary package aim: run `validate -> plan -> run -> runs list/show` on a typed spec
- Secondary scientific aim: relate input perturbations (`a`, `b`) to output behavior (`y`)

## Success criteria
- you can explain one observed trend in outputs and point to its run artifacts


## Why this tutorial matters
Each step connects the CLI workflow to scientific reasoning so you can explain not only *what* ran, but *why* results are meaningful.


## Step 1: Validate and plan


In [ ]:
%%bash
cd /Users/barak/Downloads/metamodeler_codex_scaffold_docs
PYTHONPATH=src python -m metamodeler.cli.main validate tutorials/specs/model.toy.grid.json
PYTHONPATH=src python -m metamodeler.cli.main plan tutorials/specs/model.toy.grid.json


## Step 2: Execute runs and list run IDs


In [ ]:
%%bash
cd /Users/barak/Downloads/metamodeler_codex_scaffold_docs
PYTHONPATH=src python -m metamodeler.cli.main run tutorials/specs/model.toy.grid.json
PYTHONPATH=src python -m metamodeler.cli.main runs list


## Step 3: Auto-select a recent run and inspect it


In [ ]:
import json
from pathlib import Path
import subprocess
import os

root = Path('/Users/barak/Downloads/metamodeler_codex_scaffold_docs')
reg_path = root / 'tmp/run_registry.json'
registry = json.loads(reg_path.read_text())

candidates = []
for run_id, record in registry.items():
    if isinstance(record, str):
        run_payload = json.loads((root / record).read_text())
    else:
        run_payload = record
    inputs_path = run_payload.get('inputs_path', '')
    if inputs_path.startswith('tmp/tutorials/toy_store/runs/'):
        candidates.append((run_payload.get('finished_at', ''), run_id))

if not candidates:
    raise RuntimeError('No tutorial toy runs found. Run Step 2 first.')

run_id = sorted(candidates)[-1][1]
print('Using run id:', run_id)

env = dict(os.environ)
env['PYTHONPATH'] = 'src'
subprocess.run(
    ['python', '-m', 'metamodeler.cli.main', 'runs', 'show', run_id],
    cwd=root,
    env=env,
    check=False,
)


## Step 4: Visualize response surface (graphic)


In [ ]:
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

runs_root = Path('/Users/barak/Downloads/metamodeler_codex_scaffold_docs/tmp/tutorials/toy_store/runs')
rows = []
for run_dir in sorted(p for p in runs_root.iterdir() if p.is_dir()):
    inp = json.loads((run_dir / 'inputs.json').read_text())
    out = json.loads((run_dir / 'outputs.json').read_text())
    y0 = out['y']['y'][0]
    rows.append((inp['a'], inp['b'], y0))

arr = np.array(rows, dtype=float)
plt.figure(figsize=(5, 4))
sc = plt.scatter(arr[:, 0], arr[:, 1], c=arr[:, 2], s=120, cmap='viridis')
plt.xlabel('a')
plt.ylabel('b')
plt.title('Toy model first output component y[0]')
plt.colorbar(sc, label='y[0]')
plt.grid(True, alpha=0.3)
plt.show()


## Scientific checkpoint
- Describe one monotonic trend you see in the plot.
- Explain whether the trend matches your expectation from the toy equation.


## Common mistakes
- Running from the wrong working directory.
- Forgetting `PYTHONPATH=src` in module mode.
